# g25 stereo의 pose 미관측 269개 — 고정 warm 전용
새 Colab CPU 런타임에서 root가 고정한 source/input/기존 235MiB cache ZIP을 준비합니다. 다른 원본·cache로 바꾸거나 cold로 대체하지 않습니다.

정확 목표는 Pulling_A_Rope__9_의147개(0~148 중51·80 제외)와 Blow_A_Kiss__14_의122개(0~121)입니다. RGB 이름 대응의 미관측이며 원본 pose 결측 판정이 아닙니다. 모두 train이고 학습 release는 없습니다.

기존235MiB를 byte0부터 로컬 재해제하고 suffix만 새로 읽습니다. 원본 끝까지 잔여121729956bytes(약116.09MiB), 최대465개의256KiB media range와 약1397prepare HTTP는 추정치입니다. 실제 시간·해제 상한 내 목표 완료를 보장하지 않습니다.

Prepare body192MiB / 1916HTTP / 1200초 / logical352MiB / 반환decode2400MiB / 파일4MiB / 일반출력32MiB를 유지합니다. 초기 metadata128KiB·8HTTP·30초와 별도cache export60초·localIO1152MiB는 구분합니다. 결과ZIP33MiB, cache384MiB, import/replay localIO708MiB입니다.

관측 크기로269 JSON은약23MiB 예상일 뿐입니다. 상한에 닿으면 부분 결과를 회수하고 자동 반복하지 않습니다. 관측목록16000entry가 truncate돼도 목표 추출은 계속하며 목표완료와 전체EOF/CRC를 구분합니다.

Pose의 제한 metadata pre-open timeout retry정책은 기존대로입니다. Hidden auth/media retry는0이고 RGB 인덱싱 retry0와 구분합니다.

가용 RAM/임시 disk는 각각2GiB 이상 확인합니다. 업로드·인증·로컬 검사·prepare·export 비용을 구분하세요. 원본 cache를 보존하고 회수 셀에서 결과 ZIP과 확장cache ZIP을 각각 영구 보관합니다. 실패·중단 후에는 회수 셀만 실행합니다.


In [ ]:
# root가 커밋 후 source/입력 SHA와 revision·job 경로를 고정합니다.
SOURCE_ZIP_SHA256 = ""
SOURCE_REVISION = ""
NOTEBOOK_REVISION = ""
INPUT_ZIP_SHA256 = "ef22cd2953a70a9c718b525402150cbc44a5a9ae514d096ae5de3fcb17789700"
TARGETS_SHA256 = "fa0bdf8458a931d8adb8f5dfdf7dc2c2a2dcbc8e989e97680fd8c6720b65b41c"
CACHE_ZIP_SHA256 = "f902b25aadb1932c4c734795592e122061f8d30410bdeed5bb3866528ebff437"
JOB_ID = "pose_missing269_001"
SOURCE_ROOT = "/content/pose_missing269_001-source"
INPUT_ROOT = "/content/pose_missing269_001-inputs"
CACHE_INPUT = "/content/pose_missing269_001-cache.zip"
WORKSPACE = "/content/pose_missing269_001-work"
RECOVERY_ZIP = "/content/pose_missing269_001_preparation_result.zip"


In [ ]:
import hashlib, io, json, re, stat, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files
assert re.fullmatch("[a-f0-9]{64}", SOURCE_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", INPUT_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", TARGETS_SHA256)
assert re.fullmatch("[a-f0-9]{64}", CACHE_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{40}", SOURCE_REVISION)
assert not Path(WORKSPACE).exists() and not Path(WORKSPACE).is_symlink(), "소비된 job: 준비/scan 재실행 금지"
import psutil, shutil, time
assert psutil.virtual_memory().available >= 2*1024**3, "가용 RAM2GiB 필요"
assert shutil.disk_usage(Path(SOURCE_ROOT).parent).free >= 2*1024**3, "임시 disk2GiB 필요"
print("1/3: 고정 source ZIP을 선택하세요.")
source_uploaded = files.upload()
print("2/3: 고정 input ZIP을 선택하세요.")
input_uploaded = files.upload()
print("3/3: 고정 235MiB v4 cache ZIP을 선택하세요. 누락 시 cold 대체하지 않습니다.")
cache_uploaded = files.upload()
local_started = time.monotonic()
def selected(uploaded, pin, label):
    rows = [b for b in uploaded.values() if hashlib.sha256(b).hexdigest() == pin]
    assert len(rows) == 1, label + " ZIP 누락 또는 고정 SHA 불일치"
    return rows[0]
def inspect_zip(raw, maximum):
    assert len(raw) <= maximum, "압축 ZIP 크기 상한"
    payloads, directories, names = {}, set(), set()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        rows = z.infolist()
        assert len(rows) <= 2048 and sum(i.file_size for i in rows) <= maximum, "ZIP 해제 상한"
        for item in rows:
            name = item.filename.rstrip("/")
            path = PurePosixPath(name)
            assert name and name == path.as_posix() and not path.is_absolute()
            assert ".." not in path.parts and "\\" not in name and all(":" not in part for part in path.parts), "ZIP 경로 오류"
            assert name.casefold() not in names, "ZIP 중복 경로"
            names.add(name.casefold())
            kind = stat.S_IFMT(item.external_attr >> 16)
            assert kind in (0, stat.S_IFDIR if item.is_dir() else stat.S_IFREG), "ZIP 링크/특수파일 금지"
            for parent in path.parents:
                if str(parent) != ".": directories.add(parent.as_posix())
            data = z.read(item)  # 모든 member CRC를 추출 전에 검사
            assert len(data) == item.file_size
            if item.is_dir(): directories.add(name)
            else: payloads[name] = data
    assert not set(payloads) & directories, "ZIP 파일/폴더 충돌"
    return payloads, directories
# source/input/cache 세 ZIP을 검증한 뒤 준비 경로를 만듭니다.
source_data = inspect_zip(selected(source_uploaded, SOURCE_ZIP_SHA256, "source"), 64*1024**2)
input_data = inspect_zip(selected(input_uploaded, INPUT_ZIP_SHA256, "input"), 8*1024**2)
assert source_data[0]["SOURCE_REVISION.txt"].decode("utf-8").strip() == SOURCE_REVISION
input_config = json.loads(input_data[0]["config.json"])
targets = json.loads(input_data[0]["targets.json"])
assert len(targets) == 269 and targets == input_config["job"]["exact_targets"] == input_config["job"]["includes"]
assert hashlib.sha256(json.dumps(targets, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest() == TARGETS_SHA256 == input_config["targets_content_sha256"]
cache_raw = selected(cache_uploaded, CACHE_ZIP_SHA256, "cache")
assert CACHE_ZIP_SHA256 == input_config["cache_zip_sha256"] and len(cache_raw) == input_config["cache_zip_bytes"] <= 384*1024**2
with zipfile.ZipFile(io.BytesIO(cache_raw)) as z:
    assert set(z.namelist()) == {"prefix.bin", "manifest.json"} and len(z.infolist()) == 2
    assert all(i.compress_type == zipfile.ZIP_STORED and not i.extra and not i.comment for i in z.infolist())
    assert z.getinfo("manifest.json").file_size <= 1024**2 and sum(i.file_size for i in z.infolist()) <= 384*1024**2
    assert z.testzip() is None, "cache CRC 불일치"
    cache_manifest = json.loads(z.read("manifest.json"))
    expected = input_config["job"]["cache_expectation"]
    assert {k:cache_manifest[v] for k,v in (("manifest_sha256","content_sha256"),("prefix_sha256","prefix_sha256"),("prefix_bytes","prefix_bytes"))} == expected
    assert z.getinfo("prefix.bin").file_size == expected["prefix_bytes"]
cache_path = Path(CACHE_INPUT).absolute()
for part in (cache_path, *cache_path.parents):
    assert not part.is_symlink() and not getattr(part, "is_junction", lambda:False)()
assert not cache_path.is_relative_to(Path(SOURCE_ROOT).absolute()) and not cache_path.is_relative_to(Path(INPUT_ROOT).absolute())
cache_exists = cache_path.exists()
if cache_exists:
    assert cache_path.is_file() and cache_path.stat().st_size == len(cache_raw)
    h = hashlib.sha256()
    with cache_path.open("rb") as stream:
        while data := stream.read(65536): h.update(data)
    assert h.hexdigest() == CACHE_ZIP_SHA256, "기존 cache 불일치: 덮어쓰기 금지"
assert time.monotonic()-local_started <= 60, "업로드 후 로컬 검사60초 제한"
def existing_matches(destination, contents):
    target = Path(destination).absolute()
    for part in (target, *target.parents):
        assert not part.is_symlink() and not getattr(part, "is_junction", lambda:False)(), "추출 경로 링크 금지"
    if not target.exists(): return False
    assert target.is_dir(), "추출 대상이 디렉터리가 아님"
    files_seen, dirs_seen = set(), set()
    for count, path in enumerate(target.rglob("*")):
        assert count < 4096 and not path.is_symlink() and not getattr(path, "is_junction", lambda:False)()
        relative = path.relative_to(target).as_posix()
        if path.is_dir(): dirs_seen.add(relative)
        else:
            assert path.is_file() and relative in contents[0], "기존 추출본의 다른 파일"
            expected = contents[0][relative]
            assert path.stat().st_size == len(expected) and path.read_bytes() == expected, "기존 추출본 바이트 불일치"
            files_seen.add(relative)
    assert files_seen == set(contents[0]) and dirs_seen == contents[1], "기존 추출본 파일/폴더 집합 불일치"
    return True
source_path, input_path = Path(SOURCE_ROOT).absolute(), Path(INPUT_ROOT).absolute()
assert not source_path.is_relative_to(input_path) and not input_path.is_relative_to(source_path)
source_exists = existing_matches(SOURCE_ROOT, source_data)
input_exists = existing_matches(INPUT_ROOT, input_data)
def materialize(destination, contents, exists):
    if exists: return  # 정확히 같은 기존 추출본만 읽기 재사용
    target = Path(destination)
    target.mkdir(parents=True, exist_ok=False)
    for name in sorted(contents[1], key=lambda s:len(PurePosixPath(s).parts)):
        (target/name).mkdir(exist_ok=True)
    for name, data in contents[0].items():
        with (target/name).open("xb") as stream: stream.write(data)
materialize(SOURCE_ROOT, source_data, source_exists)
materialize(INPUT_ROOT, input_data, input_exists)
if not cache_exists:
    with cache_path.open("xb") as stream: stream.write(cache_raw)
assert time.monotonic()-local_started <= 60, "업로드 후 로컬 준비60초 제한"
sys.dont_write_bytecode = True
sys.path.insert(0, SOURCE_ROOT)
del source_uploaded, input_uploaded, cache_uploaded, source_data, input_data, cache_raw


In [ ]:
# 네트워크 전, 고정 source·cache policy·목표·예산을 검사합니다.
import json, sys
from pathlib import Path
sys.dont_write_bytecode = True
for name, module in tuple(sys.modules.items()):
    if name == "rewind_archive" or name.startswith("rewind_archive."):
        loaded = getattr(module, "__file__", None)
        assert loaded and Path(loaded).resolve().is_relative_to(Path(SOURCE_ROOT).resolve()), "이전 rewind_archive가 로드됨: 새 CPU 런타임에서 다시 준비하세요"
sys.path.insert(0, SOURCE_ROOT)
from rewind_archive.workflow import make_preflight
from rewind_archive.prefix_cache import input_spec
from rewind_archive.cache_policy import TAIL352_POLICY
config = json.loads((Path(INPUT_ROOT)/"config.json").read_bytes())
JOB = config["job"]
JOB["job_id"] = JOB_ID
JOB["cache_input"] = input_spec(CACHE_INPUT, policy=TAIL352_POLICY)
PREFLIGHT = make_preflight(JOB, {})
print(PREFLIGHT["summary"])


In [ ]:
# 1회만 실행합니다. 중단/실패 후에는 회수 셀만 실행합니다.
import json, sys
from pathlib import Path
assert not Path(WORKSPACE).exists(), "소비된 job: 재실행 금지"
from google.colab import auth
auth.authenticate_user()
import google.auth
import requests
from google.auth.transport.requests import AuthorizedSession, Request
from rewind_archive.request_policy import validate_no_hidden_retries
from urllib3.util.retry import Retry
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
def session_factory():
    auth_http = requests.Session()
    try:
        auth_http.trust_env = False
        for adapter in auth_http.adapters.values():
            adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
        session = AuthorizedSession(credentials, max_refresh_attempts=0, auth_request=Request(session=auth_http))
    except BaseException:
        auth_http.close()
        raise
    close_outer = session.close
    def close_all():
        try:
            close_outer()
        finally:
            auth_http.close()
    session.close = close_all
    try:
        session.trust_env = False
        for adapter in session.adapters.values():
            adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
        # 초기 metadata 호출 전 내부 auth transport까지 검사한다.
        validate_no_hidden_retries(session)
    except BaseException:
        session.close()
        raise
    return session
from rewind_archive.workflow import run_workflow_job
OUTPUTS = Path(WORKSPACE)/"outputs"
OUTPUTS.mkdir(parents=True, exist_ok=False)
RESULT_ROOT, RECORD = run_workflow_job(JOB, PREFLIGHT, {}, OUTPUTS, session_factory)
print({k:RECORD.get(k) for k in ("status","stop_reason","completed_targets","missing_targets","targets_complete")})


In [ ]:
# 독립 회수: 인증/원본 요청/run은 다시 호출하지 않습니다.
import hashlib, json, sys, zipfile
from pathlib import Path
from google.colab import files
def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while data := stream.read(65536): h.update(data)
    return h.hexdigest()
job_root = Path(WORKSPACE)/"outputs"/("archive_job_"+JOB_ID)
if not (job_root/"job.json").is_file():
    target = Path(RECOVERY_ZIP)
    assert target.name.endswith("_preparation_result.zip")
    if not target.exists():
        info = {"job_id":JOB_ID, "targets_sha256":TARGETS_SHA256, "source_revision":SOURCE_REVISION,
            "notebook_revision":NOTEBOOK_REVISION, "status":"preparation_or_auth_failed_execution_unverified",
            "job_report_present":False, "train_ready":False}
        with zipfile.ZipFile(target, "x", zipfile.ZIP_STORED) as z:
            z.writestr("recovery.json", json.dumps(info, ensure_ascii=False, sort_keys=True))
    with zipfile.ZipFile(target) as z:
        assert z.testzip() is None
        info = json.loads(z.read("recovery.json"))
        assert info["job_id"] == JOB_ID and info["targets_sha256"] == TARGETS_SHA256
    print("준비 결과 ZIP", str(target), file_sha(target))
    files.download(str(target))
else:
    assert (Path(SOURCE_ROOT)/"SOURCE_REVISION.txt").read_text().strip() == SOURCE_REVISION
    sys.dont_write_bytecode = True
    sys.path.insert(0, SOURCE_ROOT)
    from rewind_archive.workflow import review_run, save_result_zip, save_cache_zip
    from rewind_archive.contracts import PreparationError
    checked = review_run(job_root)
    extra = job_root/"pilot"/"pose269_review.json"
    extra.parent.mkdir(exist_ok=True)
    if not extra.exists():
        with extra.open("x", encoding="utf-8") as stream: json.dump(checked, stream, ensure_ascii=False, indent=2)
    cache = None
    if (job_root/"pilot"/"cache"/"manifest.json").is_file():
        try:
            cache, cache_report = save_cache_zip(job_root)
            print({"cache_zip":str(cache), "cache_export":cache_report})
        except PreparationError as exc:
            print({"cache_export_failed":exc.reason, "original_cache_preserved":True})
    # 결과 ZIP은 cache body와 별도이며 JSON·hash·missing·종료 이유·관측목록을 포함합니다.
    result = save_result_zip(job_root)
    print("결과 ZIP", str(result), file_sha(result))
    files.download(str(result))
    if cache is not None:
        print("확장 cache ZIP", str(cache), file_sha(cache))
        files.download(str(cache))
